In [2]:
import parsedatetime
import datetime
import time
import copy
import json
import os

In [3]:
results_dir = "results"
results_files = os.listdir(results_dir)
results = {file: json.load(open(f"{results_dir}/{file}", "r")) for file in results_files}

In [4]:
constants = parsedatetime.Constants(usePyICU=True)
cal = parsedatetime.Calendar(constants=constants)
sourceTime = time.struct_time([2026, 6, 11] + [0] * 6)

In [5]:
def evaluate_dt(result):
    preds = result['individual_results']
    corr = 0
    temporal_len = 0
    for pred in preds:
        if not pred['category'] == 2:
            continue
        try:
            ref_date, _ = cal.parse(str(pred['reference']), sourceTime=sourceTime)
            pred_date, _ = cal.parse(str(pred['prediction']), sourceTime=sourceTime)
        except Exception as e:
            continue
        if not ref_date == sourceTime:
            temporal_len += 1
            if ref_date == pred_date:
                corr += 1

    return (f"{corr} / {temporal_len}", (corr / temporal_len) * 100)

In [6]:
for result in results.values():
    corr, score = evaluate_dt(result)
    print(result['model'].split('/')[-1] + '\t', corr, '\t', score)

gpt-4o-mini	 34 / 275 	 12.363636363636363
Qwen2.5-3B-Instruct	 8 / 275 	 2.909090909090909
Qwen2.5-1.5B-Instruct	 10 / 275 	 3.6363636363636362
Llama-3.2-1B-Instruct	 1 / 275 	 0.36363636363636365
Llama-3.2-3B-Instruct	 8 / 275 	 2.909090909090909


In [12]:
def temporal_analysis(results):
    header = "\\textbf{Model}\t&\t\\textbf{Accuracy}\t&\t\\textbf{Correct Answers}\t\\\\"
    print(header)
    for result in results:
        model = result['model'].split('/')[-1]
        if "gpt" in model:
            model = model.replace("gpt", "GPT")
        samples = result['individual_results']
        corr = 0
        temporal_len = 0
        for sample in samples:
            if not sample['category'] == 2:
                continue
            ref, _ = cal.parse(str(sample['reference']), sourceTime=sourceTime)
            pred, _ = cal.parse(str(sample['prediction']), sourceTime=sourceTime)

            if ref == sourceTime:
                continue
            ref_date = datetime.datetime.fromtimestamp(time.mktime(ref)).strftime("%Y-%m-%d %H:%M")
            pred_date = datetime.datetime.fromtimestamp(time.mktime(pred)).strftime("%Y-%m-%d %H:%M")
            temporal_len += 1

            if pred_date in ref_date or ref_date in pred_date or ref_date == pred_date:
                corr += 1                
        
        print(f"{model}\t&\t{(corr / temporal_len) * 100:.2f}\t&\t{corr} / {temporal_len}\t\\\\")

In [13]:
temporal_analysis(results.values())

\textbf{Model}	&	\textbf{Accuracy}	&	\textbf{Correct Answers}	\\
GPT-4o-mini	&	12.36	&	34 / 275	\\
Qwen2.5-3B-Instruct	&	2.91	&	8 / 275	\\
Qwen2.5-1.5B-Instruct	&	3.64	&	10 / 275	\\
Llama-3.2-1B-Instruct	&	0.36	&	1 / 275	\\
Llama-3.2-3B-Instruct	&	2.91	&	8 / 275	\\


In [14]:
def sort_categories(result):
    category_names = ["multi_hop", "temporal", "open_domain", "single_hop", "adversial"]
    categories = {
        i + 1: name
        for i, name in enumerate(category_names)
    }
    result_by_category = {
        name: [pred for pred in result['individual_results'] if pred['category'] == i]
        for i, name in categories.items()
    }

    return result_by_category

In [15]:
output_dir = "results_by_category"
os.makedirs(output_dir, exist_ok=True)
for result_file, result in results.items():
    _result = copy.deepcopy(result)
    _result.pop("individual_results")
    _result['by_category'] = sort_categories(result)

    with open(f"{output_dir}/{result_file}", "w") as file:
        json.dump(_result, file, indent=2)
